# 🧩 Mini-Lab: Streaming Responses

**Module 2: LLM Core Concepts** | **Duration: ~20 min** | **Type: Mini-Lab**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Understand** how autoregressive generation works (token-by-token)
2. **Understand** how streaming differs from regular API responses
3. **Implement** streaming with OpenAI's API
4. **Process** tokens as they arrive in real-time
5. **Build** interactive user experiences with streaming

## Target Concepts

| Concept | Description |
|---------|-------------|
| Autoregressive Generation | LLMs generate text one token at a time, each depending on previous tokens |
| Streaming | Receiving tokens incrementally as they're generated |

## 🧠 How LLMs Generate Text: Autoregressive Generation

**LLMs generate text one token at a time.** This is called **autoregressive generation**:

```
Prompt: "The capital of France is"

Step 1: Model predicts → "Paris" (based on prompt)
Step 2: Model predicts → "." (based on prompt + "Paris")
Step 3: Model predicts → "\n" (based on prompt + "Paris" + ".")
...and so on until a stop condition is met
```

### Why This Matters

1. **Each token depends on ALL previous tokens** - the model "reads" everything before predicting the next word
2. **Generation is sequential** - you can't generate token 5 before tokens 1-4
3. **This is why streaming is possible** - we can send each token as it's generated
4. **This is why context matters** - earlier tokens influence all later predictions

### Visualizing the Process

```
Time →
┌─────┬──────┬───────┬──────┬─────┬─────┐
│ The │ cap  │ ital  │ of   │ Fr  │ance │  ← Input (prompt)
└─────┴──────┴───────┴──────┴─────┴─────┘
                                        ↓
                                    ┌───────┐
                                    │ is    │  Token 1 (generated)
                                    └───────┘
                                            ↓
                                        ┌───────┐
                                        │ Paris │  Token 2 (generated)
                                        └───────┘
                                                ↓
                                            ┌───┐
                                            │ . │  Token 3 (generated)
                                            └───┘
```

**Streaming exposes this token-by-token generation to your application!**

## Why Streaming Matters

- **Reduced perceived latency**: Users see output immediately
- **Better UX**: Feels more interactive and responsive
- **Early termination**: Can stop generation if output is wrong
- **Real-time processing**: Process tokens as they arrive

## 1. Setup

In [6]:
import os
import time
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, Markdown, clear_output

load_dotenv()
client = OpenAI()

def md(text):
    display(Markdown(text))

print("✓ Setup complete")

✓ Setup complete


## 2. Non-Streaming vs Streaming Comparison

First, let's compare the two approaches:

In [7]:
def compare_latency(prompt):
    """Compare perceived latency between streaming and non-streaming."""
    
    print("\n📊 Latency Comparison")
    print("="*50)
    print(f"Prompt: {prompt}\n")
    
    # Non-streaming: measure time to first character
    print("1️⃣ NON-STREAMING:")
    start = time.time()
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=200,
        stream=False
    )
    first_char_time = time.time() - start
    total_time = first_char_time  # Same as total for non-streaming
    content = response.choices[0].message.content
    print(f"   Time to first character: {first_char_time:.2f}s")
    print(f"   Total time: {total_time:.2f}s")
    print(f"   Response length: {len(content)} chars")
    
    # Streaming: measure time to first token
    print("\n2️⃣ STREAMING:")
    start = time.time()
    stream = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=200,
        stream=True
    )
    
    first_token_time = None
    streamed_content = ""
    
    for chunk in stream:
        if chunk.choices[0].delta.content:
            if first_token_time is None:
                first_token_time = time.time() - start
            streamed_content += chunk.choices[0].delta.content
    
    total_time_stream = time.time() - start
    
    print(f"   Time to first character: {first_token_time:.2f}s")
    print(f"   Total time: {total_time_stream:.2f}s")
    print(f"   Response length: {len(streamed_content)} chars")
    
    # Calculate improvement
    improvement = ((first_char_time - first_token_time) / first_char_time) * 100
    print(f"\n✨ Streaming is {improvement:.0f}% faster for first character!")

compare_latency("Explain the concept of machine learning in 3 sentences.")


📊 Latency Comparison
Prompt: Explain the concept of machine learning in 3 sentences.

1️⃣ NON-STREAMING:
   Time to first character: 2.35s
   Total time: 2.35s
   Response length: 545 chars

2️⃣ STREAMING:
   Time to first character: 0.62s
   Total time: 1.41s
   Response length: 473 chars

✨ Streaming is 74% faster for first character!


## 3. Basic Streaming Implementation

In [8]:
def basic_streaming(prompt):
    """Basic streaming demonstration."""
    
    print(f"📝 Prompt: {prompt}\n")
    print("📤 Streaming response:\n")
    
    stream = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=200,
        stream=True
    )
    
    full_response = ""
    
    for chunk in stream:
        # Each chunk contains a delta (partial content)
        if chunk.choices[0].delta.content:
            content = chunk.choices[0].delta.content
            # streming with print command
            # print(content, end="", flush=True)  # Print without newline
            full_response += content

            # streming with markdown command
            # Clear and re-render for live update effect
            clear_output(wait=True)
            md(full_response)

    print("\n")  # Final newline
    
    return full_response

basic_streaming("List 5 benefits of learning programming, briefly.");

Certainly! Here are five benefits of learning programming:

1. **Problem-Solving Skills**: Programming teaches you how to break down complex problems into manageable parts, enhancing critical thinking and logical reasoning.

2. **Career Opportunities**: Proficiency in programming opens doors to various career paths in technology, data science, web development, and more, often leading to high-demand jobs.

3. **Creativity and Innovation**: Learning to code allows you to create your own applications, websites, and software, enabling you to turn ideas into reality and innovate solutions.

4. **Automation and Efficiency**: Programming skills enable you to automate repetitive tasks, increasing productivity and efficiency in both personal and professional settings.

5. **Collaboration and Communication**: Many programming projects require teamwork, fostering collaboration skills and the ability to communicate complex ideas effectively with others.

## 4. Interactive Streaming with Live Updates

In [9]:
def interactive_streaming(prompt):
    """Stream with live markdown rendering in Jupyter."""
    
    stream = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": prompt + "\n\nFormat your response in markdown."}
        ],
        max_tokens=300,
        stream=True
    )
    
    full_response = ""
    
    for chunk in stream:
        if chunk.choices[0].delta.content:
            content = chunk.choices[0].delta.content
            full_response += content
            
            # Clear and re-render for live update effect
            clear_output(wait=True)
            md(f"**Prompt:** *{prompt}*\n\n---\n\n{full_response}▌")
    
    # Final render without cursor
    clear_output(wait=True)
    md(f"**Prompt:** *{prompt}*\n\n---\n\n{full_response}")
    
    return full_response

interactive_streaming("What are the key principles of good software design?");

**Prompt:** *What are the key principles of good software design?*

---

# Key Principles of Good Software Design

Good software design is crucial for creating maintainable, scalable, and efficient applications. Here are some key principles that underpin effective software design:

## 1. **Single Responsibility Principle (SRP)**
   - A class should have one and only one reason to change, meaning it should have only one job or responsibility.

## 2. **Open/Closed Principle (OCP)**
   - Software entities (classes, modules, functions, etc.) should be open for extension but closed for modification. This promotes flexibility to enhance functionality without altering existing code.

## 3. **Liskov Substitution Principle (LSP)**
   - Objects of a superclass should be replaceable with objects of a subclass without affecting the correctness of the program. This ensures that derived classes extend the base classes without changing their behavior.

## 4. **Interface Segregation Principle (ISP)**
   - Clients should not be forced to depend on interfaces they do not use. This encourages the design of smaller, specific interfaces instead of large monolithic ones.

## 5. **Dependency Inversion Principle (DIP)**
   - High-level modules should not depend on low-level modules, but both should depend on abstractions. This reduces coupling and increases flexibility in the codebase.

## 6. **DRY (Don't Repeat Yourself)**
   - Avoid code duplication by abstracting common functionality into reusable code. This makes the system easier to manage and

## 5. Processing Streamed Content

Process tokens as they arrive for various use cases:

In [16]:
def stream_with_word_detection(prompt, watch_words):
    """Stream and detect specific words in real-time."""
    
    print(f"📝 Prompt: {prompt}")
    print(f"👀 Watching for: {watch_words}\n")
    
    stream = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=200,
        stream=True
    )
    
    full_response = ""
    found_words = []
    
    for chunk in stream:
        if chunk.choices[0].delta.content:
            content = chunk.choices[0].delta.content
            full_response += content
            
            # Check for watch words
            for word in watch_words:
                if word.lower() in full_response.lower() and word not in found_words:
                    found_words.append(word)
                    print(f"\n🎯 Found '{word}'!")
            
            print(content, end="", flush=True)
    
    print(f"\n\n📊 Summary: Found {len(found_words)}/{len(watch_words)} watch words")
    
    # Display as rendered markdown for better readability
    print("\n📋 Rendered Output:\n")
    md(full_response)
    
    return full_response, found_words

stream_with_word_detection(
    "Explain the benefits of neural networks and deep learning for AI applications.",
    ["neural", "deep", "learning", "training", "model"]
)

📝 Prompt: Explain the benefits of neural networks and deep learning for AI applications.
👀 Watching for: ['neural', 'deep', 'learning', 'training', 'model']

Ne
🎯 Found 'neural'!
ural networks and
🎯 Found 'deep'!
 deep
🎯 Found 'learning'!
 learning have revolutionized the field of artificial intelligence (AI) and have become fundamental components in various applications. Here are some of the key benefits they provide:

### 1. **Powerful Feature Extraction**

Deep learning
🎯 Found 'model'!
 models, particularly deep neural networks, excel at automatically learning and extracting features from raw data. Unlike traditional machine learning methods, which often require manual feature engineering, deep learning can identify complex patterns without human intervention, making it suitable for high-dimensional and unstructured data such as images, audio, and text.

### 2. **Performance in Complex Tasks**

Neural networks are capable of handling complex tasks with higher accuracy compared to c

Neural networks and deep learning have revolutionized the field of artificial intelligence (AI) and have become fundamental components in various applications. Here are some of the key benefits they provide:

### 1. **Powerful Feature Extraction**

Deep learning models, particularly deep neural networks, excel at automatically learning and extracting features from raw data. Unlike traditional machine learning methods, which often require manual feature engineering, deep learning can identify complex patterns without human intervention, making it suitable for high-dimensional and unstructured data such as images, audio, and text.

### 2. **Performance in Complex Tasks**

Neural networks are capable of handling complex tasks with higher accuracy compared to conventional algorithms. For example, they have shown impressive results in image classification (using convolutional neural networks), natural language processing (using recurrent and transformer networks), and game playing (through reinforcement learning). This capability allows for the development of robust AI applications across a variety of domains.

### 3. **Scalability**

Deep learning models can

('Neural networks and deep learning have revolutionized the field of artificial intelligence (AI) and have become fundamental components in various applications. Here are some of the key benefits they provide:\n\n### 1. **Powerful Feature Extraction**\n\nDeep learning models, particularly deep neural networks, excel at automatically learning and extracting features from raw data. Unlike traditional machine learning methods, which often require manual feature engineering, deep learning can identify complex patterns without human intervention, making it suitable for high-dimensional and unstructured data such as images, audio, and text.\n\n### 2. **Performance in Complex Tasks**\n\nNeural networks are capable of handling complex tasks with higher accuracy compared to conventional algorithms. For example, they have shown impressive results in image classification (using convolutional neural networks), natural language processing (using recurrent and transformer networks), and game playing

In [17]:
def stream_with_early_stop(prompt, stop_phrase):
    """Stream and stop when a specific phrase is detected."""
    
    print(f"📝 Prompt: {prompt}")
    print(f"🛑 Will stop at: '{stop_phrase}'\n")
    
    stream = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=500,
        stream=True
    )
    
    full_response = ""
    stopped_early = False
    
    for chunk in stream:
        if chunk.choices[0].delta.content:
            content = chunk.choices[0].delta.content
            full_response += content
            print(content, end="", flush=True)
            
            # Check for stop phrase
            if stop_phrase.lower() in full_response.lower():
                print(f"\n\n⏹️ Stopped: Found '{stop_phrase}'")
                stopped_early = True
                break
    
    if not stopped_early:
        print("\n\n✅ Completed without finding stop phrase")
    
    return full_response, stopped_early

stream_with_early_stop(
    "Count from 1 to 20, writing each number on a new line.",
    stop_phrase="10"
);

📝 Prompt: Count from 1 to 20, writing each number on a new line.
🛑 Will stop at: '10'

Sure! Here are the numbers from 1 to 20, each on a new line:

1  
2  
3  
4  
5  
6  
7  
8  
9  
10

⏹️ Stopped: Found '10'


## 6. Token Counting During Streaming

In [18]:
def stream_with_stats(prompt, max_tokens=200):
    """Stream with real-time statistics."""
    
    start_time = time.time()
    
    stream = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens,
        stream=True,
        stream_options={"include_usage": True}  # Get token counts
    )
    
    full_response = ""
    chunk_count = 0
    first_token_time = None
    usage = None
    
    for chunk in stream:
        chunk_count += 1
        
        # Capture usage from final chunk
        if chunk.usage:
            usage = chunk.usage
        
        if chunk.choices and chunk.choices[0].delta.content:
            if first_token_time is None:
                first_token_time = time.time() - start_time
            
            content = chunk.choices[0].delta.content
            full_response += content
            print(content, end="", flush=True)
    
    total_time = time.time() - start_time
    
    print(f"\n\n{'='*50}")
    print("📊 Streaming Statistics:")
    print(f"   Chunks received: {chunk_count}")
    print(f"   Time to first token: {first_token_time:.3f}s")
    print(f"   Total time: {total_time:.3f}s")
    print(f"   Characters: {len(full_response)}")
    
    if usage:
        print(f"   Prompt tokens: {usage.prompt_tokens}")
        print(f"   Completion tokens: {usage.completion_tokens}")
        tokens_per_second = usage.completion_tokens / total_time
        print(f"   Speed: {tokens_per_second:.1f} tokens/second")
    
    # Display as rendered markdown for better readability
    print(f"\n{'='*50}")
    print("📋 Rendered Output:\n")
    md(full_response)
    
    return full_response

stream_with_stats("Write a short poem about coding.");

In lines of logic, thoughts take flight,  
With nimble fingers through the night.  
A world of code, both vast and deep,  
Where dreams of pixels softly creep.

Loops that dance, and functions sing,  
In silken threads, the syntax cling.  
A spark ignites with every key,  
Awakening the mind, setting it free.

Errors whisper, then fade away,  
With every debug, brighter the day.  
A tapestry woven, by hand refined,  
In the language of numbers, the heart aligned.

So here’s to the coders, in coffee we trust,  
Crafting worlds in silence, as wondrous as dust.  
For in each line written, a story unfolds,  
In the realm of the digital, the future she holds.

📊 Streaming Statistics:
   Chunks received: 164
   Time to first token: 1.244s
   Total time: 3.797s
   Characters: 662
   Prompt tokens: 14
   Completion tokens: 161
   Speed: 42.4 tokens/second

📋 Rendered Output:



In lines of logic, thoughts take flight,  
With nimble fingers through the night.  
A world of code, both vast and deep,  
Where dreams of pixels softly creep.

Loops that dance, and functions sing,  
In silken threads, the syntax cling.  
A spark ignites with every key,  
Awakening the mind, setting it free.

Errors whisper, then fade away,  
With every debug, brighter the day.  
A tapestry woven, by hand refined,  
In the language of numbers, the heart aligned.

So here’s to the coders, in coffee we trust,  
Crafting worlds in silence, as wondrous as dust.  
For in each line written, a story unfolds,  
In the realm of the digital, the future she holds.

## 7. Building a Chat Interface with Streaming

In [19]:
class StreamingChat:
    """Simple streaming chat interface."""
    
    def __init__(self, system_prompt="You are a helpful assistant."):
        self.messages = [{"role": "system", "content": system_prompt}]
        self.client = OpenAI()
    
    def chat(self, user_message):
        """Send message and stream response."""
        
        # Add user message
        self.messages.append({"role": "user", "content": user_message})
        
        print(f"👤 You: {user_message}")
        print(f"🤖 Assistant: ", end="")
        
        # Stream response
        stream = self.client.chat.completions.create(
            model="gpt-4o-mini",
            messages=self.messages,
            max_tokens=300,
            stream=True
        )
        
        assistant_response = ""
        
        # Create a display handle for live updates
        display_handle = display(Markdown(""), display_id=True)
        
        for chunk in stream:
            if chunk.choices[0].delta.content:
                content = chunk.choices[0].delta.content
                assistant_response += content
                
                # Update the specific display without clearing other output
                display_handle.update(Markdown(assistant_response + "▌"))
        
        # Final update without cursor
        display_handle.update(Markdown(assistant_response))
        print()  # Add spacing after response
        
        # Add assistant response to history
        self.messages.append({"role": "assistant", "content": assistant_response})
        
        return assistant_response
    
    def clear_history(self):
        """Clear conversation history, keeping system prompt."""
        self.messages = [self.messages[0]]
        print("🗑️ History cleared")

chat = StreamingChat("You are a helpful coding tutor. Keep responses concise.")
chat.chat("What is a function in programming?");
chat.chat("Can you show a simple example in Python?");


👤 You: What is a function in programming?
🤖 Assistant: 

A function in programming is a reusable block of code that performs a specific task. It can take inputs (arguments), process them, and return an output. Functions help organize code, reduce redundancy, and improve readability.

Here's a simple example in Python:

```python
def add(a, b):
    return a + b

result = add(2, 3)  # result is 5
```


👤 You: Can you show a simple example in Python?
🤖 Assistant: 

Sure! Here’s a simple example of a function in Python that calculates the square of a number:

```python
def square(number):
    return number * number

result = square(5)  # result is 25
print(result)       # Output: 25
```

In this example, `square` is the function that takes one argument, `number`, and returns its square.

In [20]:
chat.chat("What about functions with parameters?");

👤 You: What about functions with parameters?
🤖 Assistant: 

Functions with parameters allow you to pass values into the function to customize its behavior. Here's an example in Python that uses parameters:

```python
def greet(name, age):
    return f"Hello, {name}! You are {age} years old."

message = greet("Alice", 30)  # message is "Hello, Alice! You are 30 years old."
print(message)                 # Output: Hello, Alice! You are 30 years old.
```

In this example, the `greet` function takes two parameters: `name` and `age`. You can call it with different arguments to create personalized greetings.

## 🎯 Summary

### Key Takeaways

1. **Streaming Benefits**
   - Faster perceived response time
   - Better user experience
   - Ability to process/stop early

2. **Implementation**
   - Set `stream=True` in API call
   - Iterate over chunks
   - Access content via `chunk.choices[0].delta.content`

3. **Common Patterns**
   - Live markdown rendering
   - Word/phrase detection
   - Early termination
   - Token counting with `stream_options`

4. **Best Practices**
   - Always use `flush=True` when printing
   - Handle `None` content in chunks
   - Track both chunks and content

### Next Steps

- **mini-model-compare**: Compare streaming across models
- **lab-llm-playground**: Build complete interactive playground
- **Module 9**: Implement streaming in production APIs